# Step 2: Data Preparation

This notebook covers:
1. Organise dataset folder structure
2. Time-based train / val split across all frames
3. Create `data.yaml`
4. Data augmentation on train set only
5. Verify final dataset structure

**Design rationale:**

Since all data originates from a single video (`sample.mp4`), a truly independent
test set cannot be constructed — any split from the same video will have some
degree of data leakage due to scene similarity.

| Split | Ratio | Purpose |
|---|---|---|
| Train | 80% | Model weight updates |
| Val | 20% | Early stopping signal and overfitting monitor only |
| Test | — | True evaluation is the unseen video at interview |

Split is **time-based** to minimise leakage from near-identical consecutive frames.

Frames without a label file are automatically treated as **background (negative) samples** by YOLO,
helping the model learn to suppress false positives.

## 2.1 Install Dependencies

In [ ]:
%pip install ultralytics albumentations pyyaml -q

## 2.2 Organise Folder Structure

```
dataset/
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

In [10]:
import os
import shutil
import random
import numpy as np
from pathlib import Path

# ── Configuration ─────────────────────────────────────
SOURCE_DIR  = "dataset/images/all"
DATASET_DIR = "dataset"
TRAIN_RATIO = 0.80
# VAL_RATIO   = 0.20 (remainder)

# ── Create folder structure ───────────────────────────
for split in ["train", "val"]:
    os.makedirs(f"{DATASET_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{DATASET_DIR}/labels/{split}", exist_ok=True)

print("Folder structure created.")

Folder structure created.


## 2.3 Time-based Train / Val Split

All frames (positive + negative) are sorted by frame number to preserve temporal order,
then split chronologically.

```
Timeline: |──────────── train (80%) ────────────|── val (20%) ──|
```

In [11]:
# ── Collect all images sorted by frame number ─────────
all_images = sorted([
    f for f in os.listdir(SOURCE_DIR)
    if f.endswith(".jpg")
])

# ── Identify positive and negative samples ────────────
positive = [
    f for f in all_images
    if os.path.exists(os.path.join(SOURCE_DIR, f.replace(".jpg", ".txt")))
]
negative = [f for f in all_images if f not in positive]

total = len(all_images)
print(f"Total images     : {total}")
print(f"Positive samples : {len(positive)} (with annotation)")
print(f"Negative samples : {len(negative)} (no annotation)")
print(f"Positive ratio   : {len(positive)/total*100:.1f}%")

# ── Time-based split ──────────────────────────────────
train_end   = int(total * TRAIN_RATIO)
train_files = all_images[:train_end]
val_files   = all_images[train_end:]

print(f"\nSplit:")
print(f"  Train : {len(train_files)} images ({len(train_files)/total*100:.0f}%)")
print(f"  Val   : {len(val_files)} images ({len(val_files)/total*100:.0f}%)")

# ── Positive sample distribution per split ────────────
pos_set = set(positive)
print(f"\nPositive samples per split:")
print(f"  Train : {sum(1 for f in train_files if f in pos_set)}")
print(f"  Val   : {sum(1 for f in val_files if f in pos_set)}")

Total images     : 336
Positive samples : 64 (with annotation)
Negative samples : 272 (no annotation)
Positive ratio   : 19.0%

Split:
  Train : 268 images (80%)
  Val   : 68 images (20%)

Positive samples per split:
  Train : 50
  Val   : 14


In [12]:
# ── Copy images and labels to respective folders ──────
def copy_files(file_list, split):
    copied_img   = 0
    copied_label = 0

    for img_file in file_list:
        label_file = img_file.replace(".jpg", ".txt")
        img_src    = os.path.join(SOURCE_DIR, img_file)
        label_src  = os.path.join(SOURCE_DIR, label_file)
        img_dst    = os.path.join(DATASET_DIR, "images", split, img_file)
        label_dst  = os.path.join(DATASET_DIR, "labels", split, label_file)

        if os.path.exists(img_src):
            shutil.copy(img_src, img_dst)
            copied_img += 1

        if os.path.exists(label_src):
            shutil.copy(label_src, label_dst)
            copied_label += 1

    print(f"{split:5s} → images: {copied_img:4d}, "
          f"positive: {copied_label:3d}, "
          f"negative: {copied_img - copied_label:3d}")

copy_files(train_files, "train")
copy_files(val_files,   "val")

train → images:  268, positive:  50, negative: 218
val   → images:   68, positive:  14, negative:  54


## 2.4 Create data.yaml

In [13]:
import yaml

# ── data.yaml ─────────────────────────────────────────
data_yaml = {
    "path" : os.path.abspath(DATASET_DIR),
    "train": "images/train",
    "val"  : "images/val",
    "nc"   : 1,
    "names": ["staff_tag"]
}

yaml_path = os.path.join(DATASET_DIR, "data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"data.yaml saved at: {yaml_path}")
print("\nContents:")
with open(yaml_path) as f:
    print(f.read())

data.yaml saved at: dataset\data.yaml

Contents:
names:
- staff_tag
nc: 1
path: c:\Users\USER\Downloads\Test\dataset
train: images/train
val: images/val



## 2.5 Data Augmentation (Train Set Only)

Applied to **all training images** — both positive and negative samples.
Val set is never augmented to preserve real-world data distribution.

| Augmentation | Reason |
|---|---|
| Horizontal Flip | Person can walk left or right |
| Vertical Flip | Person can walk up or down (overhead view) |
| Rotation ±15° | Person approaches from different angles |
| Brightness / Contrast | Indoor lighting variation |
| Gaussian Noise | Simulate sensor noise from 3D camera |

**Not used:**
- Hue shift → name tag is black & white; colour shift is irrelevant
- Heavy blur → source video already contains motion blur
- Large rotation (> 15°) → distorts the name tag beyond realistic angles

In [14]:
import cv2
import albumentations as A

random.seed(42)
np.random.seed(42)

TRAIN_IMG_DIR   = f"{DATASET_DIR}/images/train"
TRAIN_LABEL_DIR = f"{DATASET_DIR}/labels/train"
AUG_COPIES      = 3

# ── Pipeline for positive samples (with bounding box) ─
transform_with_bbox = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(noise_scale_factor=0.1, p=0.3),
], bbox_params=A.BboxParams(
    format         = "yolo",
    label_fields   = ["class_labels"],
    min_visibility = 0.3
), seed=42)

# ── Pipeline for negative samples (no bounding box) ───
transform_no_bbox = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(noise_scale_factor=0.1, p=0.3),
], seed=42)

print("Augmentation pipelines defined.")

Augmentation pipelines defined.


In [15]:
# ── Apply augmentation to all training images ─────────
train_images  = list(Path(TRAIN_IMG_DIR).glob("*.jpg"))
aug_pos_count = 0
aug_neg_count = 0

for img_path in train_images:
    label_path  = Path(TRAIN_LABEL_DIR) / img_path.with_suffix(".txt").name
    is_positive = label_path.exists()

    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    if is_positive:
        # ── Positive: augment image and bounding box ──
        bboxes, class_labels = [], []
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                class_labels.append(int(float(parts[0])))
                bboxes.append([float(x) for x in parts[1:]])

        for i in range(AUG_COPIES):
            augmented  = transform_with_bbox(
                image        = image,
                bboxes       = bboxes,
                class_labels = class_labels
            )
            aug_img    = augmented["image"]
            aug_bboxes = augmented["bboxes"]
            aug_labels = augmented["class_labels"]

            if len(aug_bboxes) == 0:
                continue

            aug_img_path   = os.path.join(TRAIN_IMG_DIR,   f"{img_path.stem}_aug{i}.jpg")
            aug_label_path = os.path.join(TRAIN_LABEL_DIR, f"{img_path.stem}_aug{i}.txt")

            cv2.imwrite(aug_img_path, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
            with open(aug_label_path, "w") as f:
                for cls, bbox in zip(aug_labels, aug_bboxes):
                    f.write(f"{cls} {' '.join(f'{x:.6f}' for x in bbox)}\n")

            aug_pos_count += 1

    else:
        # ── Negative: augment image only ──────────────
        for i in range(AUG_COPIES):
            augmented    = transform_no_bbox(image=image)
            aug_img      = augmented["image"]
            aug_img_path = os.path.join(TRAIN_IMG_DIR, f"{img_path.stem}_aug{i}.jpg")
            cv2.imwrite(aug_img_path, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
            aug_neg_count += 1

print(f"Augmentation complete.")
print(f"  Positive augmented : {aug_pos_count}")
print(f"  Negative augmented : {aug_neg_count}")
print(f"  Total new images   : {aug_pos_count + aug_neg_count}")

Augmentation complete.
  Positive augmented : 150
  Negative augmented : 654
  Total new images   : 804


## 2.6 Verify Final Dataset Structure

In [16]:
# ── Final summary ─────────────────────────────────────
for split in ["train", "val"]:
    n_images = len(list(Path(f"{DATASET_DIR}/images/{split}").glob("*.jpg")))
    n_labels = len(list(Path(f"{DATASET_DIR}/labels/{split}").glob("*.txt")))
    n_neg    = n_images - n_labels
    print(f"{split:5s} → images: {n_images:4d}, "
          f"positive: {n_labels:3d}, "
          f"negative: {n_neg:3d}")

print("\nNote: Val set is used solely for early stopping and overfitting monitoring.")
print("True generalisation will be evaluated on unseen footage during the interview.")
print("\nDataset is ready for training.")

train → images: 1072, positive: 200, negative: 872
val   → images:   68, positive:  14, negative:  54

Note: Val set is used solely for early stopping and overfitting monitoring.
True generalisation will be evaluated on unseen footage during the interview.

Dataset is ready for training.
